# GEPA Optimizer

This notebook demonstrates the GEPA (Genetic-Pareto) optimizer in DSPy 3.x.

**What You'll Learn:**
- How to use the GEPA optimizer
- How GEPA differs from other optimizers
- How to configure GEPA parameters
- When to use GEPA vs other optimizers
- How to evaluate and compare optimized models

## Setup

In [ ]:
import os
import sys
sys.path.append('../../')

import dspy
from utils import setup_default_lm, configure_dspy, print_step, print_result, print_error
from utils.datasets import get_sample_qa_data
from dotenv import load_dotenv

load_dotenv('../../.env')

try:
    lm = setup_default_lm(provider="openai", model="gpt-4o", max_tokens=500)
    configure_dspy(lm=lm)
    print_result("Language model configured successfully!", "Status")
except Exception as e:
    print_error(f"Failed to configure language model: {e}")

## Part 1: Understanding GEPA

GEPA combines three powerful techniques:
1. **Genetic Algorithms**: Evolve prompts through mutations and crossover
2. **Pareto Optimization**: Balance multiple objectives (accuracy, efficiency)
3. **Reflective Improvement**: Self-improve prompts based on failures

In [ ]:
print("Comparison with other optimizers:")
print()
print("BootstrapFewShot:")
print("  - Simple and fast")
print("  - Good for basic improvements")
print("  - Limited exploration of prompt space")
print()
print("MIPROv2:")
print("  - Bayesian optimization")
print("  - Efficient hyperparameter tuning")
print("  - Good balance of speed and quality")
print()
print("GEPA:")
print("  - Most sophisticated approach")
print("  - Best for complex tasks")
print("  - Multi-objective optimization")
print("  - Longer optimization time but better results")
print("  - Self-reflective improvement")

## Part 2: Basic GEPA Optimization

Set up a question-answering system and test its baseline performance before optimization.

In [ ]:
class QuestionAnswering(dspy.Signature):
    """Answer questions accurately and concisely."""
    question = dspy.InputField(desc="A question to answer")
    answer = dspy.OutputField(desc="A concise and accurate answer")

class QASystem(dspy.Module):
    def __init__(self):
        super().__init__()
        self.qa = dspy.ChainOfThought(QuestionAnswering)

    def forward(self, question):
        result = self.qa(question=question)
        return dspy.Prediction(answer=result.answer, reasoning=result.reasoning)

# Load and split data
all_data = get_sample_qa_data()
train_data = all_data[:8]
val_data = all_data[8:]
print(f"Training examples: {len(train_data)}")
print(f"Validation examples: {len(val_data)}")

In [ ]:
# Define evaluation metric
def qa_metric(example, prediction, trace=None):
    """Evaluate QA predictions. Returns a score between 0 and 1."""
    pred_answer = prediction.answer.lower().strip()
    true_answer = example.answer.lower().strip()

    if pred_answer == true_answer:
        return 1.0

    true_words = set(true_answer.split())
    pred_words = set(pred_answer.split())
    overlap = len(true_words & pred_words)

    if overlap > 0:
        return overlap / max(len(true_words), len(pred_words))

    return 0.0

# Test baseline
print("Testing baseline performance...")
baseline_system = QASystem()

baseline_scores = []
for example in val_data[:3]:
    try:
        prediction = baseline_system(question=example.question)
        score = qa_metric(example, prediction)
        baseline_scores.append(score)
        print(f"  Q: {example.question[:60]}...")
        print(f"  Expected: {example.answer}")
        print(f"  Got: {prediction.answer}")
        print(f"  Score: {score:.2f}\n")
    except Exception as e:
        print(f"  Error: {e}")
        baseline_scores.append(0.0)

avg_baseline = sum(baseline_scores) / len(baseline_scores) if baseline_scores else 0
print(f"Baseline average score: {avg_baseline:.2f}")

## Part 3: Running GEPA Optimization

GEPA optimization can take several minutes. For demonstration, we use small parameters.

In [ ]:
try:
    print("Initializing GEPA optimizer...")

    gepa_optimizer = dspy.GEPA(
        metric=qa_metric,
        population_size=4,      # Small for demo; use 10-20 in practice
        num_generations=2,      # Small for demo; use 5-10 in practice
        verbose=True
    )

    print("\nGEPA optimization steps:")
    print("1. Create initial population of prompt variants")
    print("2. Evaluate each variant on training data")
    print("3. Apply genetic operations (mutation, crossover)")
    print("4. Reflect on failures and improve")
    print("5. Repeat for specified generations")

    # Uncomment to actually run optimization:
    # optimized_system = gepa_optimizer.compile(student=QASystem(), trainset=train_data)

    print("\n[Optimization code is commented out for demo purposes]")
    print("In practice, GEPA typically improves scores by 10-30%")

except AttributeError as e:
    print_error(f"GEPA optimizer may not be available: {e}")

## Part 4: GEPA Configuration

Key parameters for tuning GEPA optimization.

In [ ]:
print("Key GEPA Parameters:")
print()
print("1. population_size (default: 10)")
print("   - Number of prompt variants per generation")
print("   - Larger = more exploration, slower optimization")
print("   - Recommended: 10-20 for most tasks")
print()
print("2. num_generations (default: 5)")
print("   - Number of evolutionary cycles")
print("   - More generations = better results, longer time")
print("   - Recommended: 5-10 for most tasks")
print()
print("3. mutation_rate (default: 0.1)")
print("   - Probability of mutating each prompt component")
print("   - Higher = more exploration, less stability")
print("   - Recommended: 0.1-0.3")
print()
print("4. crossover_rate (default: 0.5)")
print("   - Probability of combining two prompts")
print("   - Recommended: 0.5-0.7")
print()
print("5. reflection_enabled (default: True)")
print("   - Whether to use reflective improvement")
print("   - Recommended: Always True")
print()
print("6. num_threads (default: 1)")
print("   - Parallel evaluation threads")
print("   - Recommended: 4-8 for production")

## Part 5: When to Use GEPA

Choosing the right optimizer for your task.

In [ ]:
print("Use GEPA when:")
print("  - You have a complex task with multiple objectives")
print("  - You want the best possible performance")
print("  - You have time for longer optimization")
print("  - You have sufficient training data (50+ examples)")
print()
print("Use BootstrapFewShot when:")
print("  - You want quick results")
print("  - Your task is relatively simple")
print("  - You have limited training data (<20 examples)")
print()
print("Use MIPROv2 when:")
print("  - You want good balance of speed and quality")
print("  - You need hyperparameter tuning")
print("  - You have moderate training data (20-50 examples)")
print()
print("Use SIMBA when:")
print("  - You want self-reflective improvement")
print("  - You have examples of failures to learn from")
print("  - Your task benefits from iterative refinement")

## Part 6: Complete GEPA Optimization Template

Production-ready code structure for GEPA optimization.

In [ ]:
print("""
# 1. Define your module
class MyModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.predictor = dspy.ChainOfThought(MySignature)

    def forward(self, **inputs):
        result = self.predictor(**inputs)
        return dspy.Prediction(...)

# 2. Prepare data
train_data = [...]  # 50-100+ examples
val_data = [...]    # 20-30 examples

# 3. Define metric
def my_metric(example, prediction, trace=None):
    accuracy = ...  # 0-1 score
    efficiency = ...  # 0-1 score
    return (accuracy + efficiency) / 2

# 4. Configure GEPA
optimizer = dspy.GEPA(
    metric=my_metric,
    population_size=15,
    num_generations=7,
    mutation_rate=0.2,
    crossover_rate=0.6,
    reflection_enabled=True,
    num_threads=4,
    verbose=True
)

# 5. Optimize
optimized = optimizer.compile(student=MyModule(), trainset=train_data)

# 6. Evaluate
scores = [my_metric(ex, optimized(**ex.inputs())) for ex in val_data]
print(f"Average score: {sum(scores) / len(scores):.3f}")

# 7. Save
optimized.save("models/my_optimized_model")
""")

## Summary

**Key Takeaways:**
1. GEPA combines genetic algorithms with Pareto optimization
2. Best for complex tasks requiring sophisticated optimization
3. Key parameters: `population_size`, `num_generations`, `mutation_rate`
4. Reflection enables self-improvement from failures
5. Longer optimization time but better final results
6. Use multi-objective metrics for best results
7. Requires more training data than simpler optimizers